In [ ]:
import random
from datasets import load_dataset, Dataset, IterableDataset

# ====================================================================================
# 📝 데이터셋 정보 및 실습 목표
# 데이터셋명: mncai/WildChat_korean
# 의미: 와일드챗(WildChat)의 한국어 대화 로그 데이터입니다.
# 설명: 사용자들 간의 대화 기록과, 해당 대화가 얼마나 유해하거나 (Toxic),
#       혐오 발언(Hate), 또는 괴롭힘(Harassment)에 해당하는지를 AI 모델이 분석한 메타데이터가 포함되어 있습니다.
# 실습 목표: 대화 내용을 분석하여, 각 대화가 얼마나 위험한지 '위험도 지수'를 계산해 보고,
#           어떤 유형의 유해 콘텐츠가 가장 많이 발견되는지 탐색하는 AI 감지기 시뮬레이션을 해봅시다!
# ====================================================================================

# --- 설정 상수 ---
DATASET_ID = "mncai/WildChat_korean"
# 분석할 샘플의 개수 (메모리 절약과 빠른 학습을 위해 상위 100개만 사용합니다!)
SAMPLE_COUNT = 100
# -----------------

def load_dataset_safe(dataset_id: str, split: str, sample_count: int):
    """
    데이터셋 로드를 시도하며, 스트리밍 실패 시 일반 모드로 폴백(Fallback)합니다.
    """
    print("🛡️ [1/3] 데이터셋을 안전하게 로딩합니다...")
    
    # 1. 스트리밍 로드 시도 (가장 빠르고 효율적인 방법입니다!)
    try:
        dataset = load_dataset(dataset_id, split=split, streaming=True)
        print("✅ 성공: 스트리밍 모드 (IterableDataset)로 데이터셋을 로드했습니다. ✨")
        return dataset
    except Exception as e:
        print(f"⚠️ 경고: 스트리밍 로딩에 실패했습니다 ({e}). 일반 모드로 전환합니다.")
        
        # 2. 스트리밍 실패 시, 일반 모드로 작은 샘플만 다운로드합니다.
        try:
            # 일반 모드에서는 특정 스플릿을 지정하고 streaming=False로 설정합니다.
            dataset = load_dataset(dataset_id, split=split, streaming=False)
            print(f"✅ 성공: 일반 모드 (Dataset)로 상위 {sample_count}개 샘플을 로드했습니다. 💪")
            return dataset
        except Exception as e_fallback:
            print(f"❌ 치명적 오류: 데이터셋 로드에 실패했습니다. {e_fallback}")
            return None


def calculate_danger_score(sample):
    """
    샘플의 전체 대화 로그를 분석하여 '종합 위험도 지수(Danger Score)'를 계산합니다.
    (OpenAI Moderation 스코어들을 합산하여 계산합니다.)
    """
    # 기본 점수는 0점부터 시작합니다.
    total_danger_score = 0.0
    flagged_issues = []
    
    # 📣 Openai Moderation 결과가 있는지 확인합니다.
    if 'openai_moderation' not in sample or not sample['openai_moderation']:
        return 0.0, "분석 불가: 모더레이션 데이터가 없습니다."

    moderation_data = sample['openai_moderation']
    
    # 🔍 어떤 위협 요소들이 있는지 반복문으로 탐색합니다.
    # 핵심 위험 요소만 뽑아와서 점수에 반영합니다.
    
    # 1. 혐오 발언 (Hate) 점수 체크
    hate_score = moderation_data.get('category_scores', {}).get('hate', 0.0)
    if hate_score > 0.1:
        total_danger_score += hate_score * 1.5  # 혐오 발언은 가중치를 줍니다!
        flagged_issues.append(f"🔥 혐오 발언 (Hate): {hate_score:.2f}")
    
    # 2. 괴롭힘 (Harassment) 점수 체크
    harassment_score = moderation_data.get('category_scores', {}).get('harassment', 0.0)
    if harassment_score > 0.1:
        total_danger_score += harassment_score * 1.2
        flagged_issues.append(f"😠 괴롭힘 (Harassment): {harassment_score:.2f}")
        
    # 3. 성적인 콘텐츠 (Sexual) 점수 체크
    sexual_score = moderation_data.get('category_scores', {}).get('sexual', 0.0)
    if sexual_score > 0.1:
        total_danger_score += sexual_score * 1.3
        flagged_issues.append(f"🔞 성적인 콘텐츠 (Sexual): {sexual_score:.2f}")
        
    # 4. 일반적인 유해성 (General Toxicity)
    # 만약 전반적인 유해성 점수(detoxify_moderation의 toxicity)를 사용하고 싶다면 여기에 추가 가능합니다.
    
    if not flagged_issues:
        return 0.0, "평온함: 위험도가 감지되지 않았습니다."
    
    return total_danger_score, "\n  - " + "\n  - ".join(flagged_issues)


# ====================================================================================
# ✨ 메인 실습 함수
# ====================================================================================

def run_ai_tutor_script():
    print("\n" + "=" * 80)
    print("🤖 어서 오세요! AI 윤리 감지기 실습을 시작해 볼까요? 🎓")
    print("==============================================================================\n")
    
    # 1. 데이터 로드 (스트리밍 안전 로더 사용)
    # WildChat_korean 데이터셋의 'train' 스플릿을 사용합니다.
    dataset_iterator = load_dataset_safe(DATASET_ID, split='train', sample_count=SAMPLE_COUNT)
    
    if dataset_iterator is None:
        print("\n🚨 데이터 로드 실패로 인해 실습을 종료합니다.")
        return

    # 2. 샘플 데이터 추출 및 반복 준비 (Constraint 9 적용)
    # 전체 데이터셋을 반복하기 전에, 상위 K개의 샘플만 리스트로 가져와서 분석에 사용합니다.
    # (이는 메모리 효율적이고, 모든 실습의 기반이 됩니다.)
    print(f"🔍 [2/3] 분석할 샘플 상위 {SAMPLE_COUNT}개를 준비합니다...")
    try:
        # list(dataset_iterator.take(K)) 패턴을 사용합니다.
        sampled_dataset_list = list(dataset_iterator.take(SAMPLE_COUNT))
    except Exception as e:
        print(f"❌ 샘플 추출 중 오류 발생: {e}")
        return

    # 3. AI 분석 시뮬레이션 실행
    toxic_report = []
    for i, sample in enumerate(sampled_dataset_list):
        # 대화 ID와 원본 대화 내용을 가져와서 추적합니다.
        convo_id = sample.get('conversation_id', 'N/A')
        
        # Danger Score 계산 함수 호출!
        score, issues = calculate_danger_score(sample)
        
        # 결과 저장
        toxic_report.append({
            'id': convo_id,
            'score': score,
            'issues': issues,
            'content': sample['conversation']
        })

    # 4. 결과 출력 및 인사이트 도출 (창의적 사용 예시)
    print("\n" + "=" * 80)
    print("✨ [3/3] 💡 AI 감지기 분석 결과 요약 리포트 💡")
    print("=" * 80)
    
    # 결과 데이터를 점수가 높은 순서로 정렬합니다.
    toxic_report.sort(key=lambda x: x['score'], reverse=True)
    
    # 최고 위험 샘플만 출력하여 결과를 시각적으로 보여줍니다.
    print(f"\n🚀 총 {len(toxic_report)}개의 샘플을 분석했습니다. (상위 {SAMPLE_COUNT}개)")
    print(f"📊 상위 위험 샘플의 평균 위험도 점수: {sum(r['score'] for r in toxic_report) / len(toxic_report):.2f}점")
    print("-" * 80)
    
    # 가장 위험한 상위 5개 샘플을 사용자에게 보여줍니다.
    print("🚨 [Top 5] 위험도가 가장 높은 대화 로그 (위협 탐지!) 🚨")
    
    for i, report in enumerate(toxic_report[:5]):
        score = report['score']
        issues = report['issues']
        
        print(f"\n--- 💡 샘플 {i+1}: Conversation ID ({report['id']}) ---")
        print(f"   📈 종합 위험도 지수: {score:.2f}점 (높을수록 위험)")
        print(f"   🚩 감지된 유해 요소:\n  {issues}")
        
        # 가장 첫 번째 메시지를 예시로 출력합니다.
        if report['content']:
            first_message = report['content'][0]
            print(f"\n   💬 (탐지된 대화 샘플 예시 - 1번째 턴):")
            print(f"      [{first_message['role']}]: {first_message['content'][:40]}...")
        else:
             print("   ⚠️ 대화 내용을 가져올 수 없습니다.")
        
    print("\n" + "=" * 80)
    print("🎉 실습을 완료했습니다! 축하드려요! 🎊")
    print("이 과정을 통해 실제 대화 데이터가 어떤 종류의 위협(혐오, 괴롭힘 등)을 포함할 수 있는지 파악했습니다.")
    print("이 데이터를 활용하여 '필터링 시스템'이나 '위험 콘텐츠 경고' AI를 개발할 수 있습니다!")

if __name__ == "__main__":
    run_ai_tutor_script()